In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, glob
os.system('apt-get install unrar -qq')
src_path = '/content/drive/MyDrive/multisource/MultiSource Files'
ext_path = '/content/data/extracted'
os.makedirs(ext_path, exist_ok=True)
for f in sorted(os.listdir(src_path)):
    if f.endswith('.rar'):
        fname = f.replace('.rar', '')
        dest = os.path.join(ext_path, fname)
        os.makedirs(dest, exist_ok=True)
        os.system(f'unrar x -o+ "{os.path.join(src_path, f)}" "{dest}/"')
        print(f'done: {f}')
vzip = os.path.join(src_path, 'validation for multi source.zip')
vdest = os.path.join(ext_path, 'validation')
os.makedirs(vdest, exist_ok=True)
with zipfile.ZipFile(vzip, 'r') as z:
    z.extractall(vdest)
print('val done')
for c in glob.glob(f'{src_path}/*.csv'):
    os.system(f'cp "{c}" /content/data/')
    print(f'cp {os.path.basename(c)}')

In [ ]:
import numpy as np
import cv2
from scipy.ndimage import minimum_filter, uniform_filter
from scipy.stats import gaussian_kde
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
def is_valid_img(fn):
    if fn.startswith('._') or fn.startswith('.'): return False
    return fn.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))
def spatial_filter(img, ksz=3):
    mf = uniform_filter(img.astype(np.float64), size=ksz)
    mnf = minimum_filter(mf, size=ksz)
    return mnf.astype(np.uint8)
def extract_lung(img, thresh=None):
    filt = spatial_filter(img, ksz=3)
    if thresh is None:
        thresh, _ = cv2.threshold(filt, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        thresh = max(thresh * 0.5, 20)
    msk = np.zeros_like(filt, dtype=np.uint8)
    msk[filt >= thresh] = 1
    cnts, _ = cv2.findContours(msk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(msk)
    cv2.drawContours(filled, cnts, -1, 1, thickness=cv2.FILLED)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    filled = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, k)
    return filled, filt
def crop_lung(img, msk, sz=256):
    coords = np.where(msk > 0)
    if len(coords[0]) == 0:
        return cv2.resize(img, (sz, sz)), 0
    ymin, ymax = coords[0].min(), coords[0].max()
    xmin, xmax = coords[1].min(), coords[1].max()
    p=5
    ymin=max(0,ymin-p); ymax=min(img.shape[0],ymax+p)
    xmin=max(0,xmin-p); xmax=min(img.shape[1],xmax+p)
    cr = img[ymin:ymax, xmin:xmax]
    return cv2.resize(cr, (sz, sz)), np.sum(msk)
def kds_sampling(sls, n=8):
    areas=[]; idxs=[]
    for i, sp in enumerate(sls):
        im = cv2.imread(sp, cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        m, _ = extract_lung(im)
        areas.append(np.sum(m)); idxs.append(i)
    areas = np.array(areas, dtype=np.float64)
    idxs = np.array(idxs)
    if len(areas) < n: return idxs.tolist()
    valid = areas > 0
    if np.sum(valid) < n: return np.linspace(0, len(sls)-1, n, dtype=int).tolist()
    va=areas[valid]; vi=idxs[valid]
    try: kde = gaussian_kde(va, bw_method='scott')
    except:
        sel = np.linspace(0, len(vi)-1, n, dtype=int)
        return vi[sel].tolist()
    dens = kde(va)
    so = np.argsort(va); sd = dens[so]
    cdf = np.cumsum(sd); cdf = cdf/cdf[-1]
    picked = []
    for i in range(n):
        lo=i/n; hi=(i+1)/n; mid=(lo+hi)/2
        iv = np.where((cdf>=lo)&(cdf<hi))[0]
        if len(iv)==0: picked.append(np.argmin(np.abs(cdf-mid)))
        else:
            best=iv[np.argmin(np.abs(cdf[iv]-mid))]
            picked.append(best)
    picked = list(dict.fromkeys(picked))
    while len(picked) < n:
        for idx in np.linspace(0, len(so)-1, n*2, dtype=int):
            if idx not in picked:
                picked.append(idx)
                if len(picked)==n: break
    picked = picked[:n]
    final = vi[so[picked]]
    return sorted(final.tolist())
print('preprocessing functions ready')

In [ ]:
ext_path = '/content/data/extracted'
TR_SRC = {'covid': ['covid1', 'covid2'], 'non-covid': ['non-covid1', 'non-covid2', 'non-covid3']}
VAL_BASE = os.path.join(ext_path, 'validation', 'val')
def get_scans(sources, bpath, nm):
    out = []
    for ln, flds in sources.items():
        lb = 1 if ln=='covid' else 0
        for fld in flds:
            fp = os.path.join(bpath, fld)
            if not os.path.exists(fp): continue
            for sn in sorted(os.listdir(fp)):
                sp = os.path.join(fp, sn)
                if not os.path.isdir(sp) or sn.startswith('_'): continue
                sls = sorted([os.path.join(sp, f) for f in os.listdir(sp) if is_valid_img(f)])
                if len(sls)>=5: out.append({'scan_id': f'{fld}_{sn}', 'slices': sls, 'label': lb, 'label_name': ln})
    print(f'{nm}: {len(out)} (c={sum(1 for s in out if s["label"]==1)} nc={sum(1 for s in out if s["label"]==0)})')
    return out
def get_val_scans(vb):
    out=[]
    for ln in ['covid','non-covid']:
        lb=1 if ln=='covid' else 0
        lp=os.path.join(vb, ln)
        if not os.path.exists(lp): continue
        for sn in sorted(os.listdir(lp)):
            sp=os.path.join(lp,sn)
            if not os.path.isdir(sp) or sn.startswith('_'): continue
            sls=sorted([os.path.join(sp,f) for f in os.listdir(sp) if is_valid_img(f)])
            if len(sls)>=5: out.append({'scan_id':f'val_{sn}','slices':sls,'label':lb,'label_name':ln})
    print(f'val: {len(out)} (c={sum(1 for s in out if s["label"]==1)} nc={sum(1 for s in out if s["label"]==0)})')
    return out
tr_scans = get_scans(TR_SRC, ext_path, 'train')
vl_scans = get_val_scans(VAL_BASE)
def run_preproc(scans, outbase, nm):
    os.makedirs(os.path.join(outbase,'covid'), exist_ok=True)
    os.makedirs(os.path.join(outbase,'non-covid'), exist_ok=True)
    done=[]
    for sc in tqdm(scans, desc=nm):
        sel = kds_sampling(sc['slices'], n=8)
        imgs=[]
        for idx in sel:
            im=cv2.imread(sc['slices'][idx], cv2.IMREAD_GRAYSCALE)
            if im is None: continue
            m,_=extract_lung(im)
            cr,_=crop_lung(im,m)
            imgs.append(cr)
        if len(imgs)==0: continue
        while len(imgs)<8: imgs.append(imgs[-1])
        sd=os.path.join(outbase, sc['label_name'], sc['scan_id'])
        os.makedirs(sd, exist_ok=True)
        for i,im in enumerate(imgs[:8]):
            cv2.imwrite(os.path.join(sd, f'slice_{i:02d}.png'), im)
        done.append({'scan_id':sc['scan_id'],'label':sc['label'],'label_name':sc['label_name'],'path':sd})
    print(f'{nm}: {len(done)}')
    return done
run_preproc(tr_scans, '/content/data/preprocessed/train', 'tr')
run_preproc(vl_scans, '/content/data/preprocessed/val', 'vl')
print('done')

In [ ]:
import subprocess
subprocess.run(['pip','install','timm','albumentations','--quiet'])
import os,gc,copy
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (f1_score,roc_auc_score,confusion_matrix,classification_report,roc_curve,precision_recall_curve,auc,accuracy_score)
from torch.cuda.amp import autocast, GradScaler
import timm
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import shutil
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(dev)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
tr_cov=pd.read_csv('/content/data/train_covid.csv')
tr_ncov=pd.read_csv('/content/data/train_non_covid.csv')
vl_cov=pd.read_csv('/content/data/validation_covid.csv')
vl_ncov=pd.read_csv('/content/data/validation_non_covid.csv')
def mk_srcmap(cc, nc):
    mp={}
    for _,r in cc.iterrows(): mp[r['ct_scan_name']]=int(r['data_centre'])
    for _,r in nc.iterrows(): mp[r['ct_scan_name']]=int(r['data_centre'])
    return mp
tr_srcmap=mk_srcmap(tr_cov,tr_ncov)
vl_srcmap=mk_srcmap(vl_cov,vl_ncov)

n_src=4
src_cnts=[328, 330, 330, 234]
src_tot=sum(src_cnts)  # 1222
src_freqs=[c/src_tot for c in src_cnts]
print(f'FIXED cnts={src_cnts} freqs={[round(f,4) for f in src_freqs]}')
# Should print: [0.2684, 0.2700, 0.2700, 0.1915]

In [ ]:
IMG_MEAN=[0.485,0.456,0.406]; IMG_STD=[0.229,0.224,0.225]
tr_tfm = A.Compose([A.Resize(256,256),A.HorizontalFlip(p=0.5),A.ShiftScaleRotate(shift_limit=0.2,scale_limit=0.2,rotate_limit=30,p=0.5),A.HueSaturationValue(hue_shift_limit=20,sat_shift_limit=20,val_shift_limit=20,p=0.5),A.RandomBrightnessContrast(brightness_limit=0.2,contrast_limit=0.2,p=0.5),A.CoarseDropout(max_holes=8,max_height=32,max_width=32,p=0.2),A.Normalize(mean=IMG_MEAN,std=IMG_STD),ToTensorV2()])
vl_tfm = A.Compose([A.Resize(256,256),A.Normalize(mean=IMG_MEAN,std=IMG_STD),ToTensorV2()])
class CTDataset(Dataset):
    def __init__(self, scans, tfm=None): self.scans=scans; self.tfm=tfm
    def __len__(self): return len(self.scans)
    def __getitem__(self, i):
        info=self.scans[i]
        fls=sorted([f for f in os.listdir(info['path']) if f.endswith('.png') and not f.startswith('.')])
        ims=[]
        for f in fls[:8]:
            im=cv2.imread(os.path.join(info['path'],f),cv2.IMREAD_GRAYSCALE)
            im=cv2.cvtColor(im,cv2.COLOR_GRAY2RGB)
            if self.tfm: im=self.tfm(image=im)['image']
            ims.append(im)
        while len(ims)<8: ims.append(ims[-1])
        return (torch.stack(ims[:8],dim=0), torch.tensor(info['label'],dtype=torch.float32), torch.tensor(info['source'],dtype=torch.long))
class EffB7(nn.Module):
    def __init__(self, pt=True, nsrc=4):
        super().__init__()
        self.bb = timm.create_model('tf_efficientnet_b7', pretrained=pt, num_classes=0)
        fd = self.bb.num_features
        self.cls = nn.Sequential(nn.Dropout(0.3), nn.Linear(fd, 1))
        self.src_cls = nn.Sequential(nn.Dropout(0.3), nn.Linear(fd, nsrc))
    def forward(self, x):
        bs,nsl,C,H,W = x.shape
        feats=[]
        for i in range(nsl): feats.append(self.bb(x[:,i]))
        feat = torch.stack(feats, dim=1).mean(dim=1)
        return self.cls(feat).squeeze(-1), self.src_cls(feat)
class LALoss(nn.Module):
    def __init__(self, freqs, n=4):
        super().__init__()
        f=torch.tensor(freqs,dtype=torch.float32)
        self.register_buffer('lp', torch.log(f))
    def forward(self, logits, tgt): return F.cross_entropy(logits + self.lp, tgt)
def do_train(mdl,ldr,opt,closs,sloss,sclr,dv,g):
    mdl.train()
    tot=0.0; ps=[]; ls=[]
    for ims,lbls,srcs in tqdm(ldr,desc='tr',leave=False):
        ims=ims.to(dv); lbls=lbls.to(dv); srcs=srcs.to(dv)
        opt.zero_grad()
        with autocast():
            co,so = mdl(ims)
            lc=closs(co,lbls); ls_=sloss(so,srcs)
            loss = lc + g*ls_
        sclr.scale(loss).backward()
        sclr.step(opt); sclr.update()
        tot += loss.item()*ims.size(0)
        ps.extend(torch.sigmoid(co).detach().cpu().numpy()); ls.extend(lbls.cpu().numpy())
    return tot/len(ldr.dataset), f1_score(np.array(ls),(np.array(ps)>=0.5).astype(int))
def do_val(mdl,ldr,closs,sloss,dv,g):
    mdl.eval()
    tot=0.0; ps=[]; ls=[]
    with torch.no_grad():
        for ims,lbls,srcs in tqdm(ldr,desc='vl',leave=False):
            ims=ims.to(dv); lbls=lbls.to(dv); srcs=srcs.to(dv)
            with autocast():
                co,so = mdl(ims)
                lc=closs(co,lbls); ls_=sloss(so,srcs)
                loss=lc+g*ls_
            tot+=loss.item()*ims.size(0)
            ps.extend(torch.sigmoid(co).cpu().numpy()); ls.extend(lbls.cpu().numpy())
    p=np.array(ps); l=np.array(ls)
    f1=f1_score(l,(p>=0.5).astype(int))
    try: auc_=roc_auc_score(l,p)
    except: auc_=0.0
    return tot/len(ldr.dataset),f1,auc_,p,l
print('model + functions ready')

In [ ]:
def build_scanlist(bp, srcmap, pfx=''):
    out=[]
    for ln in ['covid','non-covid']:
        lb=1 if ln=='covid' else 0
        lp=os.path.join(bp,ln)
        if not os.path.exists(lp): continue
        for sid in sorted(os.listdir(lp)):
            sp=os.path.join(lp,sid)
            if not os.path.isdir(sp): continue
            pngs=[f for f in os.listdir(sp) if f.endswith('.png')]
            if len(pngs)==0: continue
            sname=sid.replace(pfx,'') if pfx else sid
            src=srcmap.get(sname, srcmap.get(sid,-1))
            if src==-1 and '_' in sid:
                alt=sid.split('_',1)[-1]
                src=srcmap.get(alt,-1)
            out.append({'scan_id':sid,'label':lb,'label_name':ln,'path':sp,'source':src if src!=-1 else 0})
    return out
tr_data=build_scanlist('/content/data/preprocessed/train', tr_srcmap)
vl_data=build_scanlist('/content/data/preprocessed/val', vl_srcmap, pfx='val_')
print(f'tr={len(tr_data)} vl={len(vl_data)}')
print(f'tr srcs: {dict(sorted(Counter(s["source"] for s in tr_data).items()))}')
print(f'vl srcs: {dict(sorted(Counter(s["source"] for s in vl_data).items()))}')

In [ ]:
GAMMA = 0.5

tr_ldr=DataLoader(CTDataset(tr_data,tr_tfm),batch_size=10,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
vl_ldr=DataLoader(CTDataset(vl_data,vl_tfm),batch_size=1,shuffle=False,num_workers=2,pin_memory=True)
mdl=EffB7(pt=True,nsrc=4).to(dev)
opt=torch.optim.Adam(mdl.parameters(),lr=1e-4,weight_decay=5e-4)
c_crit=nn.BCEWithLogitsLoss()
s_crit=LALoss(src_freqs,n=4).to(dev)

# verify LA loss is using correct freqs
print(f'LA log-priors: {s_crit.lp}')
# should NOT be all equal - expect approx [-1.315, -1.309, -1.309, -1.653]

sclr=GradScaler()
best_f1=0; best_auc=0; best_wts=None
hist={'tl':[],'tf':[],'vl':[],'vf':[],'va':[]}
N_EP=8
print(f'g={GAMMA} ep={N_EP}')
for ep in range(N_EP):
    print(f'\n{ep+1}/{N_EP}')
    tl,tf=do_train(mdl,tr_ldr,opt,c_crit,s_crit,sclr,dev,g=GAMMA)
    vl,vf,va,_,_=do_val(mdl,vl_ldr,c_crit,s_crit,dev,g=GAMMA)
    hist['tl'].append(tl);hist['tf'].append(tf);hist['vl'].append(vl);hist['vf'].append(vf);hist['va'].append(va)
    print(f'  tr: l={tl:.4f} f1={tf:.4f}')
    print(f'  vl: l={vl:.4f} f1={vf:.4f} auc={va:.4f}')
    if vf>best_f1:
        best_f1=vf; best_auc=va
        best_wts=copy.deepcopy(mdl.state_dict())
        print('  ^^ best')
sv='/content/drive/MyDrive/MultiSource MultiTask LA'
os.makedirs(sv,exist_ok=True)
torch.save(best_wts,f'{sv}/effnet_best_g1.0.pth')
print(f'\nbest f1={best_f1:.4f} auc={best_auc:.4f}')

In [ ]:
mdl.load_state_dict(best_wts)
mdl.eval()
allp=[]; alll=[]
with torch.no_grad():
    for ims,lbls,srcs in vl_ldr:
        ims=ims.to(dev)
        with autocast(): co,_=mdl(ims)
        allp.extend(torch.sigmoid(co).cpu().numpy()); alll.extend(lbls.numpy())
preds=np.array(allp); labels=np.array(alll)
pcls=(preds>=0.5).astype(int)
tot_score=0.0
src_ids=sorted(set(s['source'] for s in vl_data))
print(f'g={GAMMA}')
for sid in src_ids:
    ix=[i for i,s in enumerate(vl_data) if s['source']==sid]
    y=labels[ix].astype(int); p=pcls[ix]
    fc=f1_score(y,p,pos_label=1) if (y==1).sum()>0 else 0.0
    fn=f1_score(y,p,pos_label=0) if (y==0).sum()>0 else 0.0
    avg=(fc+fn)/2; tot_score+=avg
    print(f'  s{sid}: {len(ix)}scans fc={fc:.4f} fn={fn:.4f} a={avg:.4f}')
tot_score/=len(src_ids)
print(f'score={tot_score:.4f}')

In [ ]:
save_dir = f'/content/drive/MyDrive/MultiSource MultiTask LA/submission_gamma{GAMMA}'
os.makedirs(save_dir, exist_ok=True)

mdl.load_state_dict(best_wts)
_, val_f1, val_auc, val_preds, val_labels = do_val(mdl, vl_ldr, c_crit, s_crit, dev, g=GAMMA)

vpreds = (val_preds >= 0.5).astype(int)
cm = confusion_matrix(val_labels, vpreds)
acc = accuracy_score(val_labels, vpreds)
tn, fp, fn, tp = cm.ravel()
sens = tp/(tp+fn) if (tp+fn)>0 else 0
spec = tn/(tn+fp) if (tn+fp)>0 else 0

print("=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)
print(f"Accuracy:    {acc:.4f}")
print(f"F1 Score:    {val_f1:.4f}")
print(f"AUC-ROC:     {val_auc:.4f}")
print(f"Sensitivity: {sens:.4f}")
print(f"Specificity: {spec:.4f}")
print(f"\n{classification_report(val_labels, vpreds, target_names=['Non-COVID', 'COVID'])}")

src_res = {}
for i, info in enumerate(vl_data):
    s = info['source']
    if s not in src_res: src_res[s] = {'p':[], 'l':[]}
    src_res[s]['p'].append(vpreds[i])
    src_res[s]['l'].append(int(val_labels[i]))

print("=" * 60)
print("PER-SOURCE METRIC")
print("=" * 60)

metric = 0.0
srcs = sorted(src_res.keys())
src_scores = {}
f1cs, f1ns, avgs = [], [], []

for s in srcs:
    la = np.array(src_res[s]['l'])
    pa = np.array(src_res[s]['p'])
    fc = f1_score(la, pa, pos_label=1) if (la==1).sum()>0 else 0.0
    fn_ = f1_score(la, pa, pos_label=0) if (la==0).sum()>0 else 0.0
    av = (fc + fn_) / 2
    src_scores[s] = {'fc':fc, 'fn':fn_, 'avg':av, 'n':len(la)}
    f1cs.append(fc); f1ns.append(fn_); avgs.append(av)
    metric += av
    print(f"  Source {s}: {len(la)} scans | F1_COVID={fc:.4f} | F1_NonCOVID={fn_:.4f} | Avg={av:.4f}")

metric /= len(srcs)
print(f"\nMETRIC SCORE")
print(f"  (1/4) x Sigma (F1_covid_i + F1_noncovid_i) / 2 = {metric:.4f}")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

ax = axes[0, 0]
im = ax.imshow(cm, cmap='Blues')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Non-COVID', 'COVID']); ax.set_yticklabels(['Non-COVID', 'COVID'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=18,
                color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.colorbar(im, ax=ax)

fpr, tpr, _ = roc_curve(val_labels, val_preds)
ax = axes[0, 1]
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {val_auc:.4f}')
ax.plot([0, 1], [0, 1], 'r--', alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)

precision, recall, _ = precision_recall_curve(val_labels, val_preds)
pr_auc = auc(recall, precision)
ax = axes[0, 2]
ax.plot(recall, precision, 'g-', linewidth=2, label=f'PR AUC = {pr_auc:.4f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
eps = range(1, len(hist['tl']) + 1)
ax.plot(eps, hist['tl'], 'b-o', label='Train Loss')
ax.plot(eps, hist['vl'], 'r-o', label='Val Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(eps, hist['tf'], 'b-o', label='Train F1')
ax.plot(eps, hist['vf'], 'r-o', label='Val F1')
ax.plot(eps, hist['va'], 'g-s', label='Val AUC')
ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
ax.set_title('F1 Score & AUC History', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)

ax = axes[1, 2]
x = range(len(srcs))
w = 0.25
ax.bar([i-w for i in x], f1cs, w, label='F1 COVID', color='indianred')
ax.bar(list(x), f1ns, w, label='F1 Non-COVID', color='steelblue')
ax.bar([i+w for i in x], avgs, w, label='Average', color='goldenrod')
ax.set_xticks(list(x)); ax.set_xticklabels([f'Source {s}' for s in srcs])
ax.set_ylabel('F1 Score')
ax.set_title(f'Per-Source Performance (Score: {metric:.4f})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(f'{save_dir}/evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
vrecs=[]
for i,s in enumerate(vl_data):
    vrecs.append({'scan_id':s['scan_id'],'true_label':s['label_name'],'predicted_label':'covid' if vpreds[i]==1 else 'non-covid','prediction_prob':float(val_preds[i]),'correct':int(vpreds[i]==int(val_labels[i])),'data_centre':s['source']})
pd.DataFrame(vrecs).to_csv(f'{save_dir}/val_predictions.csv',index=False)
torch.save(best_wts,f'{save_dir}/effnet_best.pth')
txt=f"""multisrc covid ct - multitask + LA loss (FIXED FREQS)
g={GAMMA} ssfl+kds 8sl/scan
src_freqs={[round(f,4) for f in src_freqs]}
tr={len(tr_data)} vl={len(vl_data)}
acc={acc:.4f} f1={val_f1:.4f} auc={val_auc:.4f}
sens={sens:.4f} spec={spec:.4f}
TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}
per-src:
"""
for s in srcs:
    sc=src_scores[s]
    txt+=f"  s{s}: fc={sc['fc']:.4f} fn={sc['fn']:.4f} a={sc['avg']:.4f}\n"
txt+=f"metric={metric:.4f}\n\nepochs:\n"
for i in range(len(hist['tl'])):
    txt+=f"  {i+1}: tl={hist['tl'][i]:.4f} tf={hist['tf'][i]:.4f} vl={hist['vl'][i]:.4f} vf={hist['vf'][i]:.4f} va={hist['va'][i]:.4f}\n"
with open(f'{save_dir}/results.txt','w') as f: f.write(txt)
print(txt)
print(f'saved to {save_dir}')